# 02 — Resource Requirements

This notebook visualises the resource configurations reported by groups: memory per node, wall-time distributions, core counts, and the confidence level of submitted evidence.

**Run `00_setup.ipynb` first.**

In [ ]:
import sys
sys.path.insert(0, '/content')
sys.path.insert(0, '')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sheets_client import (
    load_sheets, map_range_labels,
    MEMORY_LABELS, CORES_LABELS, WALL_TIME_LABELS
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

dfs = load_sheets()
job_setups = dfs.get('JobSetups', pd.DataFrame())
runtime = dfs.get('RuntimeRecords', pd.DataFrame())
print(f"JobSetups records: {len(job_setups)}")
print(f"RuntimeRecords records: {len(runtime)}")

---
## Chart 1 — Memory Range Distribution by System Type

Shows how memory requirements are distributed across different system types (institutional HPC, national HPC, departmental cluster, etc.).

**What to look for:** Are there many jobs requiring memory beyond what standard nodes provide (typically 128–256 GB)? Which system types host the high-memory workloads?

In [ ]:
SYSTEM_LABELS = {
    'hpc_institutional': 'Institutional HPC',
    'hpc_national': 'National HPC',
    'hpc_departmental': 'Departmental cluster',
    'hpc_collaborator': 'Collaborator cluster',
    'cloud': 'Cloud',
    'workstation': 'Workstation',
    'other': 'Other',
    'dont_know': "Don't know",
}

MEM_ORDER = list(MEMORY_LABELS.values())

if job_setups.empty or 'memory_range' not in job_setups.columns:
    print("No memory range data in JobSetups.")
else:
    df_m = job_setups[['system_type', 'memory_range']].copy()
    df_m = df_m.dropna(subset=['memory_range'])
    df_m = df_m.loc[df_m['memory_range'].str.strip() != '']
    df_m['mem_label'] = map_range_labels(df_m['memory_range'], MEMORY_LABELS)
    df_m['sys_label'] = df_m['system_type'].map(
        lambda x: SYSTEM_LABELS.get(str(x).strip(), str(x).strip()) if pd.notna(x) else 'Unknown'
    )

    pivot = df_m.groupby(['sys_label', 'mem_label']).size().unstack(fill_value=0)
    # Reorder columns
    present_order = [c for c in MEM_ORDER if c in pivot.columns]
    pivot = pivot[present_order]

    fig, ax = plt.subplots(figsize=(11, 5))
    pivot.plot(kind='bar', stacked=True, ax=ax, colormap='Blues')
    ax.set_xlabel('System type')
    ax.set_ylabel('Number of job setup entries')
    ax.set_title('Memory Range Distribution by System Type', fontsize=13)
    ax.legend(title='Memory per node', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()

---
## Chart 2 — Observed Wall-Time Distribution

Histogram of wall times from RuntimeRecords (hours). Only numeric, non-blank entries are plotted. This shows the actual observed runtimes that groups have documented.

**What to look for:** What is the shape of the wall-time distribution? Where are the thresholds that divide workloads into short, medium, and long jobs? The committee should use this to set wall-time limits for each QoS class.

In [ ]:
if runtime.empty or 'wall_time_hours' not in runtime.columns:
    print("No wall-time data in RuntimeRecords.")
else:
    wt = pd.to_numeric(runtime['wall_time_hours'], errors='coerce').dropna()
    wt = wt[wt > 0]

    if wt.empty:
        print("No numeric wall-time values found.")
    else:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.hist(wt, bins=40, color=sns.color_palette('muted')[0], edgecolor='white', linewidth=0.5)
        # Annotate common wall-time limits
        for limit, label in [(24, '24 h'), (72, '72 h'), (168, '7 d')]:
            if limit < wt.max():
                ax.axvline(limit, color='tomato', linestyle='--', linewidth=1.2, label=label)
        ax.set_xlabel('Wall time (hours)')
        ax.set_ylabel('Number of runtime records')
        ax.set_title('Observed Wall-Time Distribution (RuntimeRecords)', fontsize=13)
        ax.legend(title='Reference limits')
        plt.tight_layout()
        plt.show()
        print(f"Records: {len(wt)}  |  Min: {wt.min():.1f} h  |  Median: {wt.median():.1f} h  |  Max: {wt.max():.1f} h")

---
## Chart 3 — Core Count Ranges

Distribution of core count ranges from JobSetups. Shows what core allocations groups are actually using.

**What to look for:** Are most jobs small (1–32 cores) or do many require large core counts? This informs minimum node count thresholds.

In [ ]:
CORES_ORDER = list(CORES_LABELS.values())

if job_setups.empty or 'cores_range' not in job_setups.columns:
    print("No core range data in JobSetups.")
else:
    df_c = job_setups['cores_range'].dropna().loc[lambda s: s.str.strip() != '']
    core_counts = map_range_labels(df_c, CORES_LABELS).value_counts()
    # Reorder
    core_counts = core_counts.reindex([c for c in CORES_ORDER if c in core_counts.index])

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(core_counts.index, core_counts.values, color=sns.color_palette('muted')[2])
    for bar, val in zip(bars, core_counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, str(val),
                ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('CPU cores per job')
    ax.set_ylabel('Number of job setup entries')
    ax.set_title('Core Count Range Distribution (JobSetups)', fontsize=13)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

---
## Chart 4 — Evidence Confidence Levels

Distribution of evidence confidence levels across all RuntimeRecords. Levels 1–5 correspond to direct production observation through projection (see CLAUDE.md section 3).

**What to look for:** How much of the submitted data is Level 1 (directly observed) versus Level 5 (projected)? This helps the committee assess data quality and identify where clarification may be useful.

In [ ]:
EVIDENCE_LABELS = {
    '1': 'L1 — Direct observation',
    '2': 'L2 — Reproducible benchmark',
    '3': 'L3 — Historical/external',
    '4': 'L4 — Software/scaling evidence',
    '5': 'L5 — Projection',
    'dont_know': "Don't know",
}

if runtime.empty or 'evidence_level' not in runtime.columns:
    print("No evidence level data in RuntimeRecords.")
else:
    ev = (
        runtime['evidence_level']
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s != '']
        .map(lambda x: EVIDENCE_LABELS.get(x, f'Level {x}'))
        .value_counts()
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(ev.index, ev.values, color=sns.color_palette('muted')[3])
    for bar, val in zip(bars, ev.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, str(val),
                ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Evidence level')
    ax.set_ylabel('Number of runtime records')
    ax.set_title('Evidence Confidence Level Distribution (RuntimeRecords)', fontsize=13)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.show()

---
## Chart 5 — Wall Time vs CPU-Hours (scatter, log scale)

Each point is one RuntimeRecord. Axes are log-scaled. Colour indicates evidence confidence level.

**What to look for:** Are long wall-time jobs also high CPU-hour consumers? Or are some long wall-time jobs small (serial or lightly parallel)? This helps distinguish intrinsic seriality from inefficient parallelisation.

In [ ]:
if runtime.empty or 'wall_time_hours' not in runtime.columns or 'cpu_gpu_hours' not in runtime.columns:
    print("No wall-time or CPU-hour data in RuntimeRecords.")
else:
    df_sc = runtime[['wall_time_hours', 'cpu_gpu_hours', 'evidence_level']].copy()
    df_sc['wall_time_hours'] = pd.to_numeric(df_sc['wall_time_hours'], errors='coerce')
    df_sc['cpu_gpu_hours'] = pd.to_numeric(df_sc['cpu_gpu_hours'], errors='coerce')
    df_sc = df_sc.dropna(subset=['wall_time_hours', 'cpu_gpu_hours'])
    df_sc = df_sc[(df_sc['wall_time_hours'] > 0) & (df_sc['cpu_gpu_hours'] > 0)]

    if df_sc.empty:
        print("No valid (wall_time, cpu_gpu_hours) pairs found.")
    else:
        df_sc['ev_label'] = df_sc['evidence_level'].astype(str).map(
            lambda x: EVIDENCE_LABELS.get(x.strip(), f'Level {x.strip()}')
        )
        palette = sns.color_palette('Set2', n_colors=df_sc['ev_label'].nunique())
        ev_cats = sorted(df_sc['ev_label'].unique())
        color_map = {cat: palette[i] for i, cat in enumerate(ev_cats)}

        fig, ax = plt.subplots(figsize=(9, 6))
        for label, grp in df_sc.groupby('ev_label'):
            ax.scatter(
                grp['wall_time_hours'], grp['cpu_gpu_hours'],
                label=label, alpha=0.7, s=50, color=color_map[label]
            )
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('Wall time (hours, log scale)')
        ax.set_ylabel('CPU/GPU hours (log scale)')
        ax.set_title('Wall Time vs CPU/GPU Hours by Evidence Level', fontsize=13)
        ax.legend(title='Evidence level', fontsize=8, loc='upper left')
        ax.grid(True, which='both', alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f"Points plotted: {len(df_sc)}")